In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchtune.modules import RotaryPositionalEmbeddings
from PIL import Image
import tiktoken
from datasets import load_dataset
import json
import re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.version.cuda)
print(f"running on {device}")

13.0
running on cpu


In [ ]:
class MultiHeadAttentionNode(nn.Module):
    def __init__(self, d_model, num_heads, dropout = 0.2):
        assert d_model % num_heads == 0
        super().__init__()

        self.Wqkv = nn.Linear(d_model, 3 * d_model)
        self.Wo = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

        self.num_heads = num_heads
        self.d_head = int(d_model / num_heads)
        self.scale = self.d_head ** 0.5

        self.rope = RotaryPositionalEmbeddings(int(d_model//num_heads))

    def forward(self, x):
        B, N, D = x.shape
        QKV = self.Wqkv(x)
        QKV = QKV.view(B, N, 3, self.num_heads, self.d_head)
        QKV = QKV.permute(2, 0, 3, 1, 4)  # (3, B, H, N, d_head)
        Q, K, V = QKV[0], QKV[1], QKV[2]

        Q = Q.permute(0, 2, 1, 3)
        K = K.permute(0, 2, 1, 3)
        Q = self.rope(Q).permute(0, 2, 1, 3)
        K = self.rope(K).permute(0, 2, 1, 3)

        score = (Q @ K.transpose(-2, -1)) # batch H N N
        score = score / self.scale

        weights = self.dropout(F.softmax(score, dim=-1))
        out = torch.matmul(weights, V) # batch H N d_head

        out = out.transpose(1, 2).contiguous() # B N H d_head
        out = out.view(B, N, D)

        out = self.dropout(self.Wo(out))

        return out, weights

class TransformerBlock(nn.Module):
    def __init__(self, d_model, d_ff, heads, dropout = 0.2):
        super().__init__()
        self.attn = MultiHeadAttentionNode(d_model, heads, dropout=dropout)
        self.ff = nn.Sequential(*[
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            
        ])
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.ln1(x))[0])
        x = x + self.dropout(self.ff(self.ln2(x)))
        return x

class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers):
        super().__init__()

        self.embed = nn.Embedding(vocab_size, d_model)
        self.transformers = nn.Sequential(
            *[TransformerBlock(d_model, d_ff, num_heads) for i in range(num_layers)]
        )
        self.ln = nn.LayerNorm(d_model)

    def forward(self, x):
        embedded = self.embed(x)
        transformed = self.transformers(embedded)
        norm = self.ln(transformed)
        return norm
    
class MLMTrainer(nn.Module):
    def __init__(self, encoder, encoder_dim, vocab_size, mask_id, dropout = 0.2, mask_ratio = 0.15):
        super().__init__()

        self.encoder = encoder
        self.vocab_size = vocab_size
        self.decoder = nn.Linear(encoder_dim, vocab_size, bias=False)
        self.dropout = nn.Dropout(dropout)
        
        self.mask_ratio = mask_ratio
        self.mask_id = mask_id

        self.decoder.weight = self.encoder.embed.weight

    def mask_tokens(self, x):
        mask = torch.rand(x.shape, device=x.device) < self.mask_ratio
        decision_mask = torch.rand(x.shape, device=x.device)
        mask_mask = mask & (decision_mask < 0.8)
        unknown_mask = mask & (decision_mask > 0.8) & (decision_mask < 0.9)
        
        ground = x.clone()
        masked = x.clone()

        masked[mask_mask] = self.mask_id
        masked[unknown_mask] = torch.randint(0, self.vocab_size, masked[unknown_mask].shape, device = device)
        ground[~mask] = -100

        return masked, ground

    def forward(self, x):
        masked, ground = self.mask_tokens(x)
        pred = self.encoder(masked)
        return self.decoder(pred), ground